# TwinCheck demo (Colab)

Runs the Gradio demo (`src/demo/app.py`) with the **trained twin-loss verdict head** and gives a public link.

- **Fast mode (default):** rule-based claim splitting + template notes. Loads in ~2 min, works on CPU or GPU.
- **Full mode:** Qwen3-8B for claims and cited notes. Needs a **T4 GPU**; first load takes 5–10 min.

Run cells 1 → 4. Cell 4 keeps running while the demo is live; stop it to end the demo.

## 1. Setup (< 1 min)


In [ ]:
!git clone -q https://github.com/himaallu/Group-Project.git /content/twincheck
%cd /content/twincheck
!pip install -q gradio   # Colab already has torch, transformers, pandas, spaCy
!python -m spacy download en_core_web_sm   # small English NER model for entity features

## 2. Settings

In [ ]:
import os
FULL_MODE = False          # True = Qwen3-8B for claims + notes (T4 GPU, slow first load)

os.environ['TWINCHECK_EXTRACTOR'] = 'qwen' if FULL_MODE else 'split'
os.environ['TWINCHECK_EXPLAINER'] = 'qwen' if FULL_MODE else 'template'
os.environ['TWINCHECK_SPACY'] = 'en_core_web_sm'
os.environ.pop('TWINCHECK_VERDICT', None)   # keep the trained head (models/verdict_twin_s42), not the rule
!python -m src.pipeline --status

## 3. Quick check (downloads the NLI model and reranker, ~1–2 min)

In [ ]:
!python -m src.pipeline "Dubai Metro suspended all lines today because of heavy flooding." --evidence "The RTA confirmed that the Red Line and the Green Line are operating normally today despite the heavy rain." "Dubai Metro services ran on schedule across the network on Tuesday, the RTA said." | grep -E '"label"|"confidence"'

## 4. Launch the demo: open the `gradio.live` link it prints

In [ ]:
!python -m src.demo.app --share

## Inputs to paste (tested; type the evidence one snippet per line)

**1. Original:** message `Dubai Metro suspended all lines today because of heavy flooding.`
evidence:
```
The RTA confirmed that the Red Line and the Green Line are operating normally today despite the heavy rain.
Dubai Metro services ran on schedule across the network on Tuesday, the RTA said.
```
→ Contradicted, 83%

**2. Arabic twin:** message `مترو دبي أوقف جميع الخطوط اليوم بسبب الفيضانات الشديدة` with the same evidence → Contradicted, 83%

**3. Unrelated-evidence twin:** the English message with evidence:
```
The Louvre Abu Dhabi will host a new exhibition of Islamic calligraphy next spring.
A local bakery chain opened three new branches in Al Ain.
```
→ Unverified, 97%

Avoid the six "ready example" buttons: with the trained head only 2 of 6 come out as expected (the "supported" weakness, slide 12).